In [385]:
import numpy as np
import pandas as pd

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer

In [386]:
# ---------------------------------------------------
# 1. Load data
# ---------------------------------------------------
df = pd.read_csv("../data/pcos_prediction_dataset.csv")
df.head()

,Country,Age,BMI,Menstrual Regularity,Hirsutism,Acne Severity,Family History of PCOS,Insulin Resistance,Lifestyle Score,Stress Levels,Urban/Rural,Socioeconomic Status,Awareness of PCOS,Fertility Concerns,Undiagnosed PCOS Likelihood,Ethnicity,Diagnosis
0,Madagascar,26,Overweight,Regular,Yes,Severe,Yes,Yes,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,Underweight,Regular,Yes,NaN,No,Yes,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,Normal,Regular,No,Moderate,No,No,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,Normal,Irregular,No,Mild,No,No,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,Overweight,Irregular,Yes,NaN,No,No,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No


In [387]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df.head()

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis
0,Madagascar,26,Overweight,Regular,Yes,Severe,Yes,Yes,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,Underweight,Regular,Yes,NaN,No,Yes,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,Normal,Regular,No,Moderate,No,No,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,Normal,Irregular,No,Mild,No,No,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,Overweight,Irregular,Yes,NaN,No,No,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No


### Feature Engineering

In [388]:
df

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis
0,Madagascar,26,Overweight,Regular,Yes,Severe,Yes,Yes,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,Underweight,Regular,Yes,NaN,No,Yes,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,Normal,Regular,No,Moderate,No,No,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,Normal,Irregular,No,Mild,No,No,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,Overweight,Irregular,Yes,NaN,No,No,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,Guinea,28,Normal,Regular,No,Moderate,Yes,No,3,Low,Urban,Middle,Yes,Yes,0.090663,African,No
119996,Mozambique,35,Overweight,Regular,No,NaN,No,No,5,Low,Rural,High,Yes,Yes,0.167482,Asian,No
119997,Cambodia,16,Normal,Regular,No,Moderate,No,No,9,Medium,Rural,Low,Yes,Yes,0.236241,African,Yes
119998,Benin,15,Obese,Regular,Yes,NaN,Yes,Yes,1,Medium,Rural,High,No,No,0.119993,Hispanic,No


In [389]:
df.bmi.value_counts()

bmi
Normal         60080
Overweight     36038
Obese          17945
Underweight     5937
Name: count, dtype: int64

In [390]:
df["bmi"] = df["bmi"].map({
    "Underweight": 1,
    "Normal": 2,
    "Overweight": 3,
    "Obese": 4
})

In [391]:
df.menstrual_regularity.value_counts()

menstrual_regularity
Regular      83941
Irregular    36059
Name: count, dtype: int64

In [392]:
df["menstrual_regularity"] = df["menstrual_regularity"].map({
    "Regular": 0,
    "Irregular": 1
})

In [393]:
df.hirsutism.value_counts()

hirsutism
No     72039
Yes    47961
Name: count, dtype: int64

In [394]:
df["hirsutism"] = df["hirsutism"].map({
    "No": 0,
    "Yes": 1
})

In [395]:
df.acne_severity.value_counts()

acne_severity
Mild        35822
Moderate    18072
Severe       6021
Name: count, dtype: int64

In [396]:
df["acne_severity"] = df["acne_severity"].map({
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3
})

In [397]:
df

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis
0,Madagascar,26,3,0,1,3.0,Yes,Yes,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,1,0,1,NaN,No,Yes,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,2,0,0,2.0,No,No,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,2,1,0,1.0,No,No,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,3,1,1,NaN,No,No,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,Guinea,28,2,0,0,2.0,Yes,No,3,Low,Urban,Middle,Yes,Yes,0.090663,African,No
119996,Mozambique,35,3,0,0,NaN,No,No,5,Low,Rural,High,Yes,Yes,0.167482,Asian,No
119997,Cambodia,16,2,0,0,2.0,No,No,9,Medium,Rural,Low,Yes,Yes,0.236241,African,Yes
119998,Benin,15,4,0,1,NaN,Yes,Yes,1,Medium,Rural,High,No,No,0.119993,Hispanic,No


In [398]:
df.family_history_of_pcos.value_counts()

family_history_of_pcos
No     84028
Yes    35972
Name: count, dtype: int64

In [399]:
df["family_history_of_pcos"] = df["family_history_of_pcos"].map({
    "No": 0,
    "Yes": 1
})

In [400]:
df.insulin_resistance.value_counts()

insulin_resistance
No     90183
Yes    29817
Name: count, dtype: int64

In [401]:
df["insulin_resistance"] = df["insulin_resistance"].map({
    "No": 0,
    "Yes": 1
})

In [402]:
df

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis
0,Madagascar,26,3,0,1,3.0,1,1,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,1,0,1,NaN,0,1,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,2,0,0,2.0,0,0,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,2,1,0,1.0,0,0,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,3,1,1,NaN,0,0,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,Guinea,28,2,0,0,2.0,1,0,3,Low,Urban,Middle,Yes,Yes,0.090663,African,No
119996,Mozambique,35,3,0,0,NaN,0,0,5,Low,Rural,High,Yes,Yes,0.167482,Asian,No
119997,Cambodia,16,2,0,0,2.0,0,0,9,Medium,Rural,Low,Yes,Yes,0.236241,African,Yes
119998,Benin,15,4,0,1,NaN,1,1,1,Medium,Rural,High,No,No,0.119993,Hispanic,No


In [403]:
df.stress_levels.value_counts()

stress_levels
Medium    59959
Low       36118
High      23923
Name: count, dtype: int64

In [404]:
df["stress_levels"] = df["stress_levels"].map({
    "Low": 1,
    "Medium": 2,
    "High": 3
})

In [405]:
df.fertility_concerns.value_counts()

fertility_concerns
No     72205
Yes    47795
Name: count, dtype: int64

In [406]:
df["fertility_concerns"] = df["fertility_concerns"].map({
    "No": 0,
    "Yes": 1
})

In [407]:
df

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis
0,Madagascar,26,3,0,1,3.0,1,1,2,1,Rural,High,Yes,0,0.107938,Hispanic,Yes
1,Vietnam,16,1,0,1,NaN,0,1,4,3,Rural,Middle,Yes,0,0.156729,Other,No
2,Somalia,41,2,0,0,2.0,0,0,7,2,Urban,Middle,Yes,1,0.202901,Other,No
3,Malawi,27,2,1,0,1.0,0,0,10,1,Urban,High,Yes,0,0.073926,Caucasian,Yes
4,France,26,3,1,1,NaN,0,0,7,2,Urban,Middle,No,0,0.229266,Caucasian,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,Guinea,28,2,0,0,2.0,1,0,3,1,Urban,Middle,Yes,1,0.090663,African,No
119996,Mozambique,35,3,0,0,NaN,0,0,5,1,Rural,High,Yes,1,0.167482,Asian,No
119997,Cambodia,16,2,0,0,2.0,0,0,9,2,Rural,Low,Yes,1,0.236241,African,Yes
119998,Benin,15,4,0,1,NaN,1,1,1,2,Rural,High,No,0,0.119993,Hispanic,No


In [408]:
df.diagnosis.value_counts()

diagnosis
No     107405
Yes     12595
Name: count, dtype: int64

In [409]:
df["diagnosis"] = df["diagnosis"].map({
    "No": 0,
    "Yes": 1
})

In [410]:
df["urban_rural"] = df["urban/rural"].map({
    "Urban": 1,
    "Rural": 0
})

In [411]:
df["socioeconomic_status"] = df["socioeconomic_status"].map({
    "Low": 1,
    "Middle": 2,
    "High": 3
})

In [412]:
df

,country,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,urban/rural,socioeconomic_status,awareness_of_pcos,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis,urban_rural
0,Madagascar,26,3,0,1,3.0,1,1,2,1,Rural,3,Yes,0,0.107938,Hispanic,1,0
1,Vietnam,16,1,0,1,NaN,0,1,4,3,Rural,2,Yes,0,0.156729,Other,0,0
2,Somalia,41,2,0,0,2.0,0,0,7,2,Urban,2,Yes,1,0.202901,Other,0,1
3,Malawi,27,2,1,0,1.0,0,0,10,1,Urban,3,Yes,0,0.073926,Caucasian,1,1
4,France,26,3,1,1,NaN,0,0,7,2,Urban,2,No,0,0.229266,Caucasian,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,Guinea,28,2,0,0,2.0,1,0,3,1,Urban,2,Yes,1,0.090663,African,0,1
119996,Mozambique,35,3,0,0,NaN,0,0,5,1,Rural,3,Yes,1,0.167482,Asian,0,0
119997,Cambodia,16,2,0,0,2.0,0,0,9,2,Rural,1,Yes,1,0.236241,African,1,0
119998,Benin,15,4,0,1,NaN,1,1,1,2,Rural,3,No,0,0.119993,Hispanic,0,0


In [413]:
df = df.drop(columns=['country', 'awareness_of_pcos', 'urban/rural'])

In [414]:
df

,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,socioeconomic_status,fertility_concerns,undiagnosed_pcos_likelihood,ethnicity,diagnosis,urban_rural
0,26,3,0,1,3.0,1,1,2,1,3,0,0.107938,Hispanic,1,0
1,16,1,0,1,NaN,0,1,4,3,2,0,0.156729,Other,0,0
2,41,2,0,0,2.0,0,0,7,2,2,1,0.202901,Other,0,1
3,27,2,1,0,1.0,0,0,10,1,3,0,0.073926,Caucasian,1,1
4,26,3,1,1,NaN,0,0,7,2,2,0,0.229266,Caucasian,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,28,2,0,0,2.0,1,0,3,1,2,1,0.090663,African,0,1
119996,35,3,0,0,NaN,0,0,5,1,3,1,0.167482,Asian,0,0
119997,16,2,0,0,2.0,0,0,9,2,1,1,0.236241,African,1,0
119998,15,4,0,1,NaN,1,1,1,2,3,0,0.119993,Hispanic,0,0


In [415]:
#print(df["ethnicity"].unique())

In [416]:
df = pd.get_dummies(df, columns=["ethnicity"], prefix="ethnicity", dtype=int)

In [417]:
df

,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,socioeconomic_status,fertility_concerns,undiagnosed_pcos_likelihood,diagnosis,urban_rural,ethnicity_African,ethnicity_Asian,ethnicity_Caucasian,ethnicity_Hispanic,ethnicity_Other
0,26,3,0,1,3.0,1,1,2,1,3,0,0.107938,1,0,0,0,0,1,0
1,16,1,0,1,NaN,0,1,4,3,2,0,0.156729,0,0,0,0,0,0,1
2,41,2,0,0,2.0,0,0,7,2,2,1,0.202901,0,1,0,0,0,0,1
3,27,2,1,0,1.0,0,0,10,1,3,0,0.073926,1,1,0,0,1,0,0
4,26,3,1,1,NaN,0,0,7,2,2,0,0.229266,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,28,2,0,0,2.0,1,0,3,1,2,1,0.090663,0,1,1,0,0,0,0
119996,35,3,0,0,NaN,0,0,5,1,3,1,0.167482,0,0,0,1,0,0,0
119997,16,2,0,0,2.0,0,0,9,2,1,1,0.236241,1,0,1,0,0,0,0
119998,15,4,0,1,NaN,1,1,1,2,3,0,0.119993,0,0,0,0,0,1,0


In [ ]:
#df.to_csv("df_clean.csv", index=False) # save the file

### Split the dataset

In [418]:
# Train /test/validation split
from sklearn.model_selection import train_test_split
df_full_train , df_test = train_test_split(df, test_size = 0.2, 
                                           random_state = 11, stratify=df["diagnosis"])
df_train, df_val = train_test_split(df_full_train, test_size=0.25, 
                                    random_state =11,stratify=df_full_train["diagnosis"])

In [419]:
df_train = df_train.reset_index(drop =True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [420]:
df_val

,age,bmi,menstrual_regularity,hirsutism,acne_severity,family_history_of_pcos,insulin_resistance,lifestyle_score,stress_levels,socioeconomic_status,fertility_concerns,undiagnosed_pcos_likelihood,diagnosis,urban_rural,ethnicity_African,ethnicity_Asian,ethnicity_Caucasian,ethnicity_Hispanic,ethnicity_Other
0,32,2,0,0,NaN,0,0,6,2,1,0,0.203525,0,1,0,0,0,1,0
1,15,4,1,1,NaN,1,0,2,3,2,0,0.053372,0,0,0,0,0,0,1
2,16,3,1,0,NaN,0,0,10,3,3,1,0.163365,0,1,1,0,0,0,0
3,17,2,0,1,1.0,1,0,4,1,3,0,0.233393,0,1,0,0,1,0,0
4,42,4,0,1,NaN,0,0,8,3,3,1,0.181062,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23995,42,4,1,0,1.0,0,0,1,3,2,1,0.100581,0,1,0,0,0,1,0
23996,39,3,1,0,1.0,0,0,6,2,1,0,0.095556,0,0,0,0,0,1,0
23997,40,2,0,0,NaN,1,1,10,2,1,0,0.086673,1,0,1,0,0,0,0
23998,23,2,0,1,1.0,1,0,8,2,1,0,0.167516,0,0,0,1,0,0,0


In [421]:
len(df_train), len(df_val), len(df_test)

(72000, 24000, 24000)

In [422]:
# Separate features 
X_train = df_train.drop('diagnosis', axis=1)
X_val= df_val.drop('diagnosis', axis=1)
X_test = df_test.drop('diagnosis', axis=1)

In [423]:
## Target Features
y_train = df_train['diagnosis']
y_val = df_val['diagnosis']
y_test = df_test['diagnosis']

In [424]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

print(le.classes_)   

[0 1]


In [425]:
# Detect column types
numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

Numeric columns: ['age', 'bmi', 'menstrual_regularity', 'hirsutism', 'acne_severity', 'family_history_of_pcos', 'insulin_resistance', 'lifestyle_score', 'stress_levels', 'socioeconomic_status', 'fertility_concerns', 'undiagnosed_pcos_likelihood', 'urban_rural', 'ethnicity_African', 'ethnicity_Asian', 'ethnicity_Caucasian', 'ethnicity_Hispanic', 'ethnicity_Other']
Categorical columns: []


In [431]:
# 7. Model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# 8. Full pipeline
#pipeline = ImbPipeline(steps=[
 #   ("preprocessor", preprocessor),
  #  ("smote", SMOTE(random_state=42, sampling_strategy=0.5)),
   # ("model", xgb)])

In [432]:
### Define hyperparameters

param_distributions = {
    "smote__sampling_strategy": [0.3, 0.5, 0.7],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__gamma": [0, 0.1, 0.3],
    "model__scale_pos_weight": [1, 2, 3, 5]
}

In [438]:
### Create cross-validation and search object
#cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#search = RandomizedSearchCV(
    #estimator=pipeline,
    #param_distributions=param_distributions,
    #n_iter=20,
    #scoring="f1",
    #n_jobs=-1,
    #cv=cv,
    #verbose=1,
    #random_state=42,
    #refit=True,
    #error_score="raise")

In [439]:
### Fit on training data

xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [440]:
print("Best CV recall:", xgb.best_score_)
print("Best parameters:")
for k, v in xgb.best_params_.items():
    print(f"{k}: {v}")

AttributeError: 'XGBClassifier' object has no attribute 'best_score_'

In [ ]:
best_model = search.best_estimator_ # store the best model

In [ ]:
### Evaluate on Validation Set
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

y_val_prob = best_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_prob >= 0.5).astype(int)

print(classification_report(y_val, y_val_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob))
print("PR-AUC:", average_precision_score(y_val, y_val_prob))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     21481
           1       0.10      1.00      0.19      2519

    accuracy                           0.10     24000
   macro avg       0.05      0.50      0.09     24000
weighted avg       0.01      0.10      0.02     24000

Confusion matrix:
 [[    0 21481]
 [    0  2519]]
ROC-AUC: 0.5
PR-AUC: 0.10495833333333333


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.91, 0.05)

results = []
for t in thresholds:
    y_pred_t = (y_val_prob >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_val, y_pred_t, zero_division=0),
        "recall": recall_score(y_val, y_pred_t, zero_division=0),
        "f1": f1_score(y_val, y_pred_t, zero_division=0)
    })

threshold_df = pd.DataFrame(results)
print(threshold_df.sort_values("f1", ascending=False))

    threshold  precision  recall        f1
0        0.10   0.104958     1.0  0.189977
6        0.40   0.104958     1.0  0.189977
9        0.55   0.104958     1.0  0.189977
1        0.15   0.104958     1.0  0.189977
7        0.45   0.104958     1.0  0.189977
8        0.50   0.104958     1.0  0.189977
5        0.35   0.104958     1.0  0.189977
4        0.30   0.104958     1.0  0.189977
3        0.25   0.104958     1.0  0.189977
2        0.20   0.104958     1.0  0.189977
10       0.60   0.000000     0.0  0.000000
11       0.65   0.000000     0.0  0.000000
12       0.70   0.000000     0.0  0.000000
13       0.75   0.000000     0.0  0.000000
14       0.80   0.000000     0.0  0.000000
15       0.85   0.000000     0.0  0.000000
16       0.90   0.000000     0.0  0.000000
